# Crossgen Biosignature Training (Rebinned to Ariel Grid)

Train the hybrid quantum-classical model on crossgen dataset.
- **Train/Val**: TauREx (37,281 / 4,142 samples)
- **Test**: POSEIDON (685 samples)
- **Spectra**: Rebinned from 218 → 44 bins (Ariel wavelength grid overlap)

In [1]:
from crossgen_hybrid_training import (
    TrainingConfig,
    run_training_experiment,
    prepare_data,
    load_crossgen_dataset,
    TARGET_COLS,
    SAFE_AUX_FEATURE_COLS,
)

config = TrainingConfig(data_root="data")
print(f"Data root:  {config.resolved_data_root()}")
print(f"Output dir: {config.resolved_output_dir()}")
print(f"Quantum:    {config.qnn_qubits} qubits, depth {config.qnn_depth}")
print(f"Device:     {config.quantum_device}")

Data root:  /Users/michalszczesny/projects/hack4sages/codex_model/data
Output dir: /Users/michalszczesny/projects/hack4sages/codex_model/outputs/model_crossgen_rebinned
Quantum:    12 qubits, depth 2
Device:     lightning.qubit


In [2]:
labels, noisy_spectra, sigma_ppm, wavelength_um = load_crossgen_dataset(config.resolved_data_root())

print(f"Labels shape:    {labels.shape}")
print(f"Spectra shape:   {noisy_spectra.shape}")
print(f"Wavelength bins: {len(wavelength_um)} ({wavelength_um[0]:.2f} - {wavelength_um[-1]:.2f} um)")
print(f"\nSplit counts:")
print(labels["split"].value_counts().to_string())
print(f"\nTarget ranges (log10 VMR):")
for col in TARGET_COLS:
    print(f"  {col}: [{labels[col].min():.2f}, {labels[col].max():.2f}], mean={labels[col].mean():.2f}")
print(f"\nAux feature ranges:")
for col in SAFE_AUX_FEATURE_COLS:
    print(f"  {col}: [{labels[col].min():.4f}, {labels[col].max():.4f}]")

Rebinned spectra: 218 bins -> 44 bins (0.95 - 4.91 um)
Labels shape:    (42108, 21)
Spectra shape:   (42108, 44)
Wavelength bins: 44 (0.95 - 4.91 um)

Split counts:
split
train    37281
val       4142
test       685

Target ranges (log10 VMR):
  log10_vmr_h2o: [-12.00, -2.00], mean=-6.99
  log10_vmr_co2: [-12.00, -2.00], mean=-6.98
  log10_vmr_co: [-12.00, -2.00], mean=-6.99
  log10_vmr_ch4: [-12.00, -2.00], mean=-7.00
  log10_vmr_nh3: [-12.00, -2.00], mean=-7.01

Aux feature ranges:
  planet_radius_rjup: [0.7000, 1.5000]
  log_g_cgs: [2.8001, 3.7000]
  temperature_k: [500.0369, 1799.9785]
  star_radius_rsun: [0.2000, 1.3000]
  log10_sigma_ppm: [1.3010, 2.0000]


In [3]:
result = run_training_experiment(config)

Project root: /Users/michalszczesny/projects/hack4sages/codex_model
Data root: /Users/michalszczesny/projects/hack4sages/codex_model/data
Output dir: /Users/michalszczesny/projects/hack4sages/codex_model/outputs/model_crossgen_rebinned
Train batch size: 256
Eval batch size: 8192
Quantum device: lightning.qubit
Quantum width/depth: 12/2
Rebinned spectra: 218 bins -> 44 bins (0.95 - 4.91 um)
Torch device: cpu | Quantum device: lightning.qubit | Dataset devices: {'train': 'cpu', 'inner_val': 'cpu', 'test': 'cpu'}
Epoch 1/30 | Batch 1/146 | batch_loss=1.02492 | avg_loss=1.02492 | batch_time=2.23s
Epoch 1/30 | Batch 2/146 | batch_loss=1.01724 | avg_loss=1.02108 | batch_time=2.01s
Epoch 1/30 | Batch 3/146 | batch_loss=0.98215 | avg_loss=1.00810 | batch_time=2.02s
Epoch 1/30 | Batch 4/146 | batch_loss=0.99696 | avg_loss=1.00532 | batch_time=2.01s
Epoch 1/30 | Batch 5/146 | batch_loss=1.01546 | avg_loss=1.00735 | batch_time=1.89s
Epoch 1/30 | Batch 6/146 | batch_loss=0.98156 | avg_loss=1.00305

In [4]:
print(f"\nBest epoch:     {result['summary']['best_epoch']}")
print(f"Best val loss:  {result['summary']['best_inner_val_loss']:.5f}")
print(f"Test RMSE mean: {result['summary']['test_rmse_mean']:.4f}")
print(f"\nPer-target test RMSE:")
print(result['test_metrics_frame'].to_string(index=False))


Best epoch:     29
Best val loss:  0.29991
Test RMSE mean: 3.4631

Per-target test RMSE:
split        target     rmse
 test log10_vmr_h2o 3.130654
 test log10_vmr_co2 4.726046
 test  log10_vmr_co 3.283202
 test log10_vmr_ch4 3.227484
 test log10_vmr_nh3 2.948282


In [5]:
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
pred_csv = pd.read_csv(result["artifacts"]["test_predictions_csv"])

for i, col in enumerate(TARGET_COLS):
    ax = axes[i]
    true_vals = pred_csv[f"true_{col}"]
    pred_vals = pred_csv[f"pred_{col}"]
    ax.scatter(true_vals, pred_vals, alpha=0.3, s=8)
    lims = [min(true_vals.min(), pred_vals.min()), max(true_vals.max(), pred_vals.max())]
    ax.plot(lims, lims, "r--", linewidth=1)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    ax.set_title(col.replace("log10_vmr_", "").upper())
    ax.set_aspect("equal")

plt.suptitle("Test Set: Predicted vs True (log10 VMR)")
plt.tight_layout()
plt.savefig("outputs/model_crossgen_rebinned/test_scatter.png", dpi=150)
plt.show()

/var/folders/wh/9tfsnwk52msftggvvzkn8gw00000gn/T/ipykernel_66146/28750786.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
